# LangGraph Voice Translation Agent (MLX Native + Local LLM)

This notebook demonstrates a professional voice translation pipeline using:
1. **mlx-audio 0.4.0**: Native ASR (Qwen3-ASR) and TTS (Qwen3-TTS) on Apple Silicon.
2. **LangChain + Local LLM**: Qwen-based translation via an OpenAI-compatible API.

## 1. Setup and Dependencies

Ensure your environment is set up with the required libraries:
```bash
uv add langgraph mlx-audio==0.2.9 huggingface_hub langchain-openai transformers torch librosa
```

### Local LLM Requirement
This agent assumes a local OpenAI-compatible API is running (e.g., via vLLM, Ollama, or LM Studio) at `http://localhost:8000/v1`.

In [1]:
import os
import sys
from typing import TypedDict, Annotated, Dict, Optional
import asyncio
from langgraph.graph import StateGraph, END

# Add src to path if needed
sys.path.append(os.path.abspath("."))
from pipeline import VoiceTranslationPipeline

## 2. Define Agent State

In [2]:
def dict_reducer(a: Dict, b: Dict | None) -> Dict:
    if b is None:
        return a
    return {**a, **b}

class AgentState(TypedDict):
    audio_path: str
    src_lang: str
    tgt_lang: str
    out_audio_path: str
    original_text: str          # raw ASR transcription (source language)
    english_text: str           # ASR text normalized to English
    translated_text: str        # final target-language text
    ref_audio_path: Optional[str]
    # Merge dicts from multiple nodes instead of “only one value per step”
    metrics: Annotated[Dict[str, float], dict_reducer]

## 3. Implement Workflow Nodes

In [3]:
pipeline = VoiceTranslationPipeline()

# ---------------------------------------------------------------------------
# Nodes
# ---------------------------------------------------------------------------
async def stt_node(state: AgentState):
    import time
    print("--- STT NODE (MLX NATIVE + EN NORMALIZATION) ---")
    start = time.time()

    # 1) Transcribe audio (whatever language ASR outputs)
    original = pipeline.transcribe_audio(state["audio_path"])

    # 2) Normalize transcription to English
    #    This uses src_lang from state (e.g., "english", "hindi", etc.)
    en_start = time.time()
    english = pipeline.translate_text(
        original,
        state["src_lang"],  # source language of the speaker
        "english",          # normalize everything to English
    )
    end = time.time()

    return {
        "original_text": original,
        "english_text": english,
        "metrics": {
            "stt_time": en_start - start,
            "stt_to_english_time": end - en_start,
        },
    }


async def mt_node(state: AgentState):
    import time
    print("--- MT NODE (ENGLISH -> TARGET, LOCAL LLM) ---")
    start = time.time()

    # Always translate from English to target language
    translated = pipeline.translate_text(
        state["english_text"],
        "english",
        state["tgt_lang"],
    )

    metrics = dict(state.get("metrics", {}))
    metrics["mt_time"] = time.time() - start
    return {
        "translated_text": translated,
        "metrics": metrics,
    }


async def voice_clone_node(state: AgentState):
    """
    Prepare a 3-second reference clip for voice cloning, if source > 3 seconds.
    Runs concurrently with MT.
    """
    import time

    print("--- VOICE CLONE NODE (PREPARE REF CLIP) ---")
    start = time.time()

    work_dir = os.path.dirname(state["out_audio_path"]) or "."
    ref_audio_path = await asyncio.to_thread(
        pipeline.prepare_voice_clone,
        state["audio_path"],
        work_dir,
    )

    metrics = dict(state.get("metrics", {}))
    metrics["voice_clone_prep_time"] = time.time() - start
    return {
        "ref_audio_path": ref_audio_path,
        "metrics": metrics,
    }


async def tts_node(state: AgentState):
    import time
    print("--- TTS NODE (MLX NATIVE, OPTIONAL VOICE CLONE) ---")
    start = time.time()
    # the first sentence of  english_text  as the reference transcript for the first 3 seconds
    ref_en = state.get("english_text") or ""
    ref_text = ref_en.split(".")[0] if "." in ref_en else ref_en
    await pipeline.text_to_speech(
        state["translated_text"],
        state["tgt_lang"],
        state["out_audio_path"],
        ref_audio=state.get("ref_audio_path"),
        ref_text=ref_text,
    )
    metrics = dict(state.get("metrics", {}))
    metrics["tts_time"] = time.time() - start
    return {
        "metrics": metrics,
    }


async def cleanup_node(state: AgentState):
    print("--- CLEANUP NODE ---")

    # Delete temporary voice clone clip if it exists
    ref_path = state.get("ref_audio_path")
    if ref_path and os.path.exists(ref_path):
        try:
            os.remove(ref_path)
            print(f"Deleted temporary voice reference clip: {ref_path}")
        except OSError as e:
            print(f"Failed to delete ref clip {ref_path}: {e}")

    pipeline.clear_memory()
    return state

Initializing Native MLX-Audio Pipeline
Models directory: /Users/nitinkumar/Work/demo/models
Using local model: /Users/nitinkumar/Work/demo/models/asr


You are using a model of type qwen3_tts to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


Using local model: /Users/nitinkumar/Work/demo/models/tts


The tokenizer you are loading from '/Users/nitinkumar/Work/demo/models/tts' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  Initialized encoder codebooks
Loaded speech tokenizer from /Users/nitinkumar/Work/demo/models/tts/speech_tokenizer
Initializing Local Translation LLM at: http://127.0.0.1:1234/v1 with model: qwen3.5-0.8b


## 4. Assemble and Run Graph

In [4]:
# ---------------------------------------------------------------------------
# Graph wiring: STT -> (MT + voice_clone in parallel) -> TTS -> cleanup
# ---------------------------------------------------------------------------
workflow = StateGraph(AgentState)

workflow.add_node("stt", stt_node)
workflow.add_node("mt", mt_node)
workflow.add_node("voice_clone", voice_clone_node)
workflow.add_node("tts", tts_node)
workflow.add_node("cleanup", cleanup_node)

workflow.set_entry_point("stt")

workflow.add_edge("stt", "mt")
workflow.add_edge("stt", "voice_clone")

workflow.add_edge("mt", "tts")
workflow.add_edge("voice_clone", "tts")

workflow.add_edge("tts", "cleanup")
workflow.add_edge("cleanup", END)

app = workflow.compile()


In [5]:
async def run_agent():
    audio_input = "../inputs/input.wav"
    output_audio_dir = "../outputs"  # treated as directory by pipeline.text_to_speech

    initial_state: AgentState = {
        "audio_path": audio_input,
        "src_lang": "english",   # language spoken in the input audio
        "tgt_lang": "hindi",     # desired target language
        "out_audio_path": output_audio_dir,
        "original_text": "",
        "english_text": "",
        "translated_text": "",
        "ref_audio_path": None,
        "metrics": {},
    }

    print("Invoking Voice Translation Agent...")
    final_state = await app.ainvoke(initial_state)

    print(f"\nOriginal (ASR) text: {final_state['original_text']}")
    print(f"English normalized text: {final_state['english_text']}")
    print(f"Resulting Translation: {final_state['translated_text']}")
    print("Ref audio used for cloning:", final_state.get("ref_audio_path"))
    print("Metrics:", final_state["metrics"])

# If running as a script:
# asyncio.run(run_agent())
await run_agent()

Invoking Voice Translation Agent...
--- STT NODE (MLX NATIVE + EN NORMALIZATION) ---
Transcribing audio: ../inputs/input.wav
--- MT NODE (ENGLISH -> TARGET, LOCAL LLM) ---
Translating via Local LLM: english -> hindi
--- VOICE CLONE NODE (PREPARE REF CLIP) ---
Source audio duration: 9.27s
Duration > 3s; creating 3-second reference clip for voice cloning.
Wrote 3s reference clip to: ../voice_ref.wav
--- TTS NODE (MLX NATIVE, OPTIONAL VOICE CLONE) ---
Generating TTS for language: hindi (voice clone: ON)
Text: मैंने एक बहुत ही समय लगभग बनाया है कि मैं अपनी आवाज़ को विकसित कर सकता था; अब जब तक यह नहीं है, तो मैं बंद हो रहा हूँ।
Voice: af_heart
Speed: 1.0x
Language: en
ICL generation: मैंने एक बहुत ही समय लगभग बनाया है कि मैं अपनी आवा...


✅ Audio successfully generated and saving as: ../outputs/audio_000.wav
Duration:              00:00:15.280
Samples/sec:           68419.2
Prompt:                191 tokens, 35.6 tokens-per-sec
Audio:                 366720 samples, 68419.2 samples-per-sec
Real-time factor:      2.85x
Processing time:       5.36s
Peak memory usage:     7.15GB
--- CLEANUP NODE ---
Deleted temporary voice reference clip: ../voice_ref.wav
Clearing models from memory...
Memory cleanup complete.

Original (ASR) text: It took me a quite long time to develop a voice, and now that I have it, I am going to be silent.
English normalized text: It took me a quite long time to develop a voice, and now that I have it, I am going to be silent.
Resulting Translation: मैंने एक बहुत ही समय लगभग बनाया है कि मैं अपनी आवाज़ को विकसित कर सकता था; अब जब तक यह नहीं है, तो मैं बंद हो रहा हूँ।
Ref audio used for cloning: ../voice_ref.wav
Metrics: {'stt_time': 0.42958879470825195, 'stt_to_english_time': 1.1205673217773438e-05, 'm